# Task 1 — Microservices Protocol

GitHub: https://github.com/Romko-studik/apz_homework_2/tree/micro_basics

## 1. Functional Correctness
### POST Transactions

**First deposite to "alice" (+500):**

![1](./images/alice_dep.png)

**Withdrawal for "alice" (-50):**

![2](./images/alice_withdr.png)

**Deposit for "bob" (+200):**

![3](./images/bob_dep.png)

### GET Requests

**GET /user/alice — balance + transaction history:**

![get alice](./images/alice_get.png)

**GET /accounts — all balances:**

![get accounts](./images/get_accounts.png)

### Service Console Output

**facade-service logs:**

![facade logs](./images/logs_facade.png)

**logging-service logs:**

![logging logs](./images/logs_logging.png)

**counter-service logs:**

![counter logs](./images/logs_counter.png)

## 2. Performance Testing
### Scenario 1 — 10 clients × 10K transactions on own accounts

![scenario 1](./images/scenario1.png)

### Scenario 2 — 10 clients × 10K transactions on shared account

![scenario 2](./images/scenario2.png)

## 3. Analysis

From our tests we can see that while each individual request runs quite quickly (~5ms network latency), 
when scaled to 100,000 requests (10 clients × 10,000 transactions each) the total test takes 
535 seconds for scenario 1 and 545 seconds for scenario 2, netting ~187 and ~183 req/s respectively.

Counter-service is around 4x faster than logging-service at the processing level (0.01ms vs 0.04ms) since the counter only performs a simple arithmetic operation on a number, while 
logging-service stores a full transaction object into its hash map.

Since we measure request time both from facade's side and inside each service separately, we can 
clearly see that network overhead is the most time-consuming (~5ms per call) while the actual service processing is 
negligible (<0.05ms). This means the bottleneck is Docker networking rather than service logic.

Scenario 1 and 2 perform almost identically despite all 10 clients writing to the same account in scenario 2. This shows that the asyncio.Lock() in counter-service introduces minimal time difference, and the shared account balance of 100,000 confirms no updates were lost.